# 🚀 ENTRENAMIENTO LLAMA-3-8B CON LoRA (V2)

---

## 📊 Especificaciones

- **Modelo:** Meta-Llama-3-8B-Instruct (8B parámetros)
- **Técnica:** QLoRA (4-bit quantization)
- **GPU:** T4 (15GB VRAM)
- **Tiempo:** 90-120 minutos
- **Loss esperado:** 0.3-0.5

---

## 📋 ORDEN DE EJECUCIÓN (SIMPLE)

1. ✅ Paso 1: Instalar dependencias
2. ✅ Paso 2: Verificar GPU
3. ✅ Paso 3: Subir dataset
4. ✅ Paso 4: Token HuggingFace
5. ✅ Paso 5: **ABRIR TENSORBOARD** (antes de entrenar)
6. ✅ Paso 6: Entrenar modelo
7. ✅ Paso 7: Descargar adaptadores

---

%%capture

# ============================================================================
# INSTALAR DEPENDENCIAS (versiones ÚLTIMAS compatibles)
# ============================================================================

# Usar las últimas versiones (dejar que pip resuelva dependencias)
!pip install -q --upgrade transformers
!pip install -q --upgrade peft
!pip install -q --upgrade accelerate
!pip install -q --upgrade bitsandbytes
!pip install -q datasets==2.14.4
!pip install -q sentencepiece==0.1.99
!pip install -q einops==0.7.0

print("✅ Dependencias instaladas (últimas versiones compatibles)")

import torch

print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU NO disponible")
    print("⚠️  Ve a Runtime → Change runtime type → T4 GPU")

print("=" * 70)

## 📤 PASO 3: SUBIR DATASET

**Sube tu archivo `dataset_pedagogico.json`**

In [ ]:
from google.colab import files
import json

print("📤 Sube tu archivo dataset_pedagogico.json")
print("   (Haz clic en 'Choose Files')\n")

uploaded = files.upload()

# Verificar que se subió correctamente
if 'dataset_pedagogico.json' in uploaded:
    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"\n✅ Dataset cargado: {len(data)} ejemplos")
    print(f"\n📝 Primer ejemplo:")
    print(f"   Instrucción: {data[0]['instruction'][:60]}...")
    print(f"   Input: {data[0]['input'][:60]}...")
else:
    print("❌ Error: No se encontró dataset_pedagogico.json")

## 🔑 PASO 4: CONFIGURAR TOKEN HUGGINGFACE

**Llama-3 requiere token de HuggingFace**

### Pasos:
1. Ve a: https://huggingface.co/settings/tokens
2. Crea un token (Read access)
3. Acepta licencia en: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
4. Ingresa el token abajo

In [ ]:
from getpass import getpass

print("🔑 TOKEN HUGGINGFACE")
print("=" * 70)
print("1. Obtén token: https://huggingface.co/settings/tokens")
print("2. Acepta licencia: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")
print("=" * 70)
print()

# Solicitar token (se oculta al escribir)
HF_TOKEN = getpass("Ingresa tu HuggingFace token: ")

if HF_TOKEN:
    print("\n✅ Token configurado correctamente")
else:
    print("\n❌ Error: Token vacío")

## 📊 PASO 5: ABRIR TENSORBOARD (ANTES DE ENTRENAR)

**⚠️ IMPORTANTE: Ejecuta esta celda ANTES del entrenamiento**

TensorBoard se actualizará automáticamente cada 30 segundos mientras entrena.

In [ ]:
# Cargar extensión de TensorBoard
%load_ext tensorboard

# Abrir TensorBoard (se actualiza en vivo)
%tensorboard --logdir logs/llama3

print("\n📊 TensorBoard abierto")
print("   Se actualizará automáticamente cada 30 segundos")
print("   Verás la curva de loss crecer en tiempo real")

## 🏋️ PASO 6: ENTRENAR LLAMA-3-8B CON QLoRA

**Tiempo estimado:** 90-120 minutos

**Mientras entrena:**
- Mira TensorBoard arriba ↑
- La curva de loss debe bajar
- Objetivo: Loss < 0.5

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

print("=" * 70)
print("🚀 ENTRENAMIENTO LLAMA-3-8B CON QLoRA")
print("=" * 70)

# ============================================================================
# CONFIGURACIÓN DEL MODELO
# ============================================================================

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

# Configuración de cuantización 4-bit (QLoRA)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Cargar en 4-bit
    bnb_4bit_quant_type="nf4",            # Tipo de cuantización
    bnb_4bit_compute_dtype=torch.float16, # Tipo de cómputo
    bnb_4bit_use_double_quant=True,       # Doble cuantización
)

# Configuración de LoRA
lora_config = LoraConfig(
    r=16,                      # Rango de LoRA
    lora_alpha=32,             # Alpha de LoRA
    target_modules=[           # Módulos a adaptar
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,         # Dropout
    bias="none",               # Sin bias
    task_type=TaskType.CAUSAL_LM
)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./lora_model",              # Directorio de salida
    num_train_epochs=10,                    # Número de épocas
    per_device_train_batch_size=1,          # Batch size (reducido por memoria)
    gradient_accumulation_steps=8,          # Acumulación de gradientes (batch efectivo = 8)
    learning_rate=2e-4,                     # Tasa de aprendizaje
    fp16=True,                              # Precisión mixta
    logging_dir="./logs/llama3",            # 📊 Directorio de logs para TensorBoard
    logging_steps=5,                        # Guardar log cada 5 steps
    save_steps=100,                         # Guardar checkpoint cada 100 steps
    save_total_limit=2,                     # Mantener solo 2 checkpoints
    warmup_steps=30,                        # Steps de warmup
    weight_decay=0.01,                      # Regularización
    max_grad_norm=1.0,                      # Gradient clipping
    optim="paged_adamw_8bit",               # Optimizador eficiente en memoria
    report_to="tensorboard",                # 📊 Reportar a TensorBoard
)

print(f"\n📦 Modelo: {MODEL_NAME}")
print(f"🔧 LoRA: r={lora_config.r}, alpha={lora_config.lora_alpha}")
print(f"🏋️  Épocas: {training_args.num_train_epochs}")
print(f"📊 Batch efectivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📉 Learning rate: {training_args.learning_rate}")
print(f"⏱️  Tiempo estimado: 90-120 minutos")
print(f"📊 TensorBoard: logs/llama3")

# ============================================================================
# PREPARAR DATASET
# ============================================================================

print("\n📚 Preparando dataset...")

with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    """Formato optimizado para Llama-3 con tokens especiales"""
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

{example['instruction']}

{example['input']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{example['output']}<|eot_id|>"""
    return {"text": text}

# Crear dataset
dataset = Dataset.from_list(data)
dataset = dataset.map(format_instruction)

print(f"   ✅ {len(dataset)} ejemplos preparados")

# ============================================================================
# CARGAR MODELO Y TOKENIZER
# ============================================================================

print("\n🤖 Cargando Llama-3-8B con cuantización 4-bit...")
print("   (Descargando ~16GB, puede tardar 3-5 minutos)")

# Cargar tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Cargar modelo con cuantización 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True
)

print(f"   ✅ Modelo cargado en GPU (4-bit)")

# Preparar modelo para entrenamiento con cuantización
model = prepare_model_for_kbit_training(model)

# ============================================================================
# APLICAR LoRA
# ============================================================================

print("\n🔧 Aplicando adaptadores LoRA...")

model = get_peft_model(model, lora_config)

# Mostrar estadísticas
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print(f"\n📊 ESTADÍSTICAS:")
print(f"   Total: {total_params:,} parámetros")
print(f"   Entrenables: {trainable_params:,} ({trainable_percentage:.4f}%)")

# ============================================================================
# TOKENIZAR DATASET
# ============================================================================

print("\n📝 Tokenizando dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print(f"   ✅ Dataset tokenizado")

# ============================================================================
# ENTRENAR (TensorBoard se actualiza automáticamente)
# ============================================================================

print("\n" + "=" * 70)
print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 70)
print(f"⏱️  Tiempo estimado: 90-120 minutos")
print(f"📉 Loss esperado: 1.5 → 0.3-0.5")
print(f"📊 Mira TensorBoard arriba ↑ (se actualiza cada 30s)")
print()

# Crear trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# ENTRENAR (TensorBoard se actualiza automáticamente)
trainer.train()

print("\n" + "=" * 70)
print("✅ ENTRENAMIENTO COMPLETADO")
print("=" * 70)

# Mostrar loss final
final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
print(f"\n📊 Loss final: {final_loss}")

if isinstance(final_loss, float):
    if final_loss < 0.3:
        print("✅ ¡EXCELENTE! Loss <0.3 - Modelo perfectamente entrenado")
    elif final_loss < 0.5:
        print("✅ MUY BIEN! Loss <0.5 - Modelo funcionará excelente")
    elif final_loss < 0.8:
        print("⚠️  ACEPTABLE - Loss <0.8 - Puede mejorar con más épocas")
    else:
        print("❌ ALTA - Loss >0.8 - Revisar configuración")

# ============================================================================
# GUARDAR ADAPTADORES
# ============================================================================

print("\n💾 Guardando adaptadores LoRA...")

model.save_pretrained("./lora_adapters")
tokenizer.save_pretrained("./lora_adapters")

print(f"   ✅ Guardado en: ./lora_adapters")

# ============================================================================
# PROBAR MODELO
# ============================================================================

print("\n🧪 PROBANDO MODELO...")
print("=" * 70)

model.eval()

test_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Eres un asistente educativo experto. Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.<|eot_id|><|start_header_id|>user<|end_header_id|>

Explica qué es una derivada

Necesito entender el concepto de derivada<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.2
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("📝 RESPUESTA DEL MODELO:")
print("-" * 70)
print(response)
print("-" * 70)

print("\n✅ PROCESO COMPLETADO")
print("\n💡 SIGUIENTE PASO: Descargar adaptadores (Paso 7)")

## 📥 PASO 7: DESCARGAR ADAPTADORES

**Descarga los adaptadores entrenados**

In [ ]:
import shutil
from google.colab import files

print("📦 Comprimiendo adaptadores...")

# Comprimir adaptadores
shutil.make_archive('lora_adapters_llama3', 'zip', './lora_adapters')

print("✅ Adaptadores comprimidos")
print("\n📥 Descargando archivo...")

# Descargar
files.download('lora_adapters_llama3.zip')

print("\n" + "=" * 70)
print("✅ DESCARGA COMPLETADA")
print("=" * 70)
print("\n📋 SIGUIENTES PASOS:")
print("   1. Descomprime lora_adapters_llama3.zip")
print("   2. Renombra la carpeta a 'lora_adapters'")
print("   3. Copia a: agent-education/fine_tuning/lora_adapters/")
print("   4. Actualiza lora_integration.py:")
print("      base_model_name='meta-llama/Meta-Llama-3-8B-Instruct'")
print("   5. Reinicia el backend de FastAPI")
print("\n🎉 ¡Llama-3-8B listo para usar!")